# WaterMamba Inference & Benchmarking Pipeline (Google Colab)

This notebook runs **end-to-end inference and evaluation** using the official
**WaterMamba** repository (Guan et al., 2024) — a State Space Model (Mamba) for
underwater image enhancement.

**Repository (reused as-is, architecture untouched):** https://github.com/Guan-MS/WaterMamba
**Paper:** *WaterMamba: Visual State Space Model for Underwater Image Enhancement* — arXiv:2405.08419

This notebook does **not** reimplement the model. It clones the official repository and
calls its own `WaterMamba` architecture class, its own YAML config loader, and the same
padding / inference logic used in the repo's own `Mamba/test.py`. The only things this
notebook adds are Colab-compatible installation, automatic Drive/dataset wiring, batching
with a progress bar, visualization, and the UIQM/UCIQE metrics (which the original
`test.py` doesn't compute).

### What is Underwater Image Enhancement (UIE)?
Underwater photos lose contrast and color as light travels through water: red wavelengths
are absorbed first (images turn green/blue), particles scatter light and create haze, and
depth-dependent attenuation makes degradation non-uniform across a single image. UIE tries
to recover a clean, color-balanced, high-contrast version of the scene from one degraded
input photo — with no access to the true clean image at inference time.

### Why WaterMamba? CNN and Transformer limitations
- **CNN-based UIE methods** (e.g. UWCNN, FUnIE-GAN) use local convolutional kernels. They're
  efficient, but a fixed receptive field struggles to model *long-range* dependencies — e.g.
  relating a hazy patch in one corner of the image to clearer color information far away.
- **Transformer-based UIE methods** (e.g. U-Transformer variants) model long-range
  dependencies well via self-attention, but self-attention cost grows *quadratically* with
  image size, making them expensive for high-resolution underwater imagery.
- **State Space Models (SSMs)**, and specifically the **Mamba** architecture, offer
  long-range sequence modeling with **linear** computational complexity — the appeal that
  motivates WaterMamba.

### Visual State Space Models & Selective Scan
Mamba treats an image (flattened into a sequence of pixels/tokens) as a discrete-time
signal and processes it with a *selective* state-space recurrence: at each step, the model
learns input-dependent parameters that decide how much of the running hidden state to keep
versus overwrite. This lets the model act like an extremely long, content-aware convolution,
without paying attention's quadratic cost.

### SCOSS, SCCOSS, and MSFFN (WaterMamba's core building blocks)
Underwater images degrade **non-uniformly** (different regions/depths degrade differently)
and lose **color-channel** information asymmetrically (red channel loss). WaterMamba
addresses both with:

- **SCCOSS (Spatial-Channel Coordinate Omnidirectional Selective Scan):** scans the feature
  map in four spatial directions *and* four channel directions, so both "where" (pixel
  dependencies) and "which channel" (color-channel dependencies) information can flow
  globally through the selective-scan mechanism.
- **SCOSS block:** wraps SCCOSS together with an **MSFFN (Multi-Scale Feed-Forward
  Network)**, which mixes multi-scale local features and helps synchronize information
  flow across the SCCOSS branches before passing to the next block.

### Encoder – Decoder – Skip Connections
```
Raw image (H x W x 3)
        │
        ▼
  Shallow conv (feature extraction)
        │
        ▼
 ┌───────────────┐   downsample   ┌───────────────┐   downsample   ┌───────────────┐
 │  SCOSS block 1 │ ─────────────▶ │  SCOSS block 2 │ ─────────────▶ │  SCOSS block 3 │  (encoder)
 └───────┬───────┘                └───────┬───────┘                └───────┬───────┘
         │  skip                          │  skip                          │
         ▼                                ▼                                ▼
 ┌───────────────┐    upsample    ┌───────────────┐    upsample    ┌───────────────┐
 │  SCOSS block 1 │ ◀───────────── │  SCOSS block 2 │ ◀───────────── │  SCOSS block 3 │  (decoder)
 └───────┬───────┘                └───────────────┘                └───────────────┘
         │
         ▼
   Conv → residual add with raw input
         │
         ▼
   Enhanced image (H x W x 3)
```
This is the standard U-shaped encoder–decoder with skip connections: the encoder
progressively downsamples while extracting multi-scale SCOSS features, the decoder mirrors
it back up while fusing in the skip-connected encoder features, and the final output is a
**residual** — the network predicts a correction that is added back onto the original raw
image, which is why the pretrained checkpoint's `net_g_*.pth` name uses the "generator"
(`_g_`) convention from the BasicSR/Restormer training framework this repo is built on.

### Inference pipeline this notebook runs
1. Load `WaterMamba.yml` → build the exact `WaterMamba` class with those hyperparameters.
2. Load `net_g_232000.pth` weights (`checkpoint["params"]`) into that model.
3. For each raw image: pad H/W up to a multiple of 8 (required by the encoder's 3x
   downsampling), run the forward pass, crop back to the original size, clamp to [0, 1].
4. Save the enhanced image, compare against ground truth, compute metrics, export results.


## Section 2 — Repository overview

The official repo has this top-level structure:

```
WaterMamba/
├─ Mamba/
│  ├─ Options/
│  │  └─ WaterMamba.yml        <- network + training/validation config (we only need network_g)
│  ├─ test.py                  <- official inference + PSNR/SSIM script (reused below)
│  └─ utils.py                 <- load_img() / save_img() helpers used by test.py
├─ basicsr/
│  └─ models/archs/
│     └─ WaterMamba_arch.py    <- the actual `WaterMamba` nn.Module (encoder/decoder/SCOSS/MSFFN)
├─ assets/                     <- README images only, not used at inference time
├─ demo/                       <- repo's own sample input/target images (we explicitly do NOT use these)
├─ setup.py / setup.cfg        <- installs the repo as an editable local package
└─ train.sh                    <- training entry point (not used here — inference only)
```

**Where things are loaded, per the official `Mamba/test.py`:**
- **YAML config** → `Mamba/Options/WaterMamba.yml`, read with `yaml.load(...)`; the
  `network_g` section (minus its `type` key) is passed as `**kwargs` straight into
  `WaterMamba(...)`.
- **Model architecture** → `from basicsr.models.archs.WaterMamba_arch import WaterMamba`.
  Because this `basicsr` folder is *vendored inside the repo itself* (not the PyPI
  `basicsr` package), we must run with the repo root at the front of `sys.path` so this
  local copy is what actually gets imported.
- **Pretrained weights** → `torch.load(weights_path)`, then
  `model.load_state_dict(checkpoint["params"])` — the checkpoint dict stores weights under
  a `"params"` key, not the raw state dict.
- **Dataset** → arbitrary `--input_dir` / `--target_dir` folders of matching-filename PNG/JPG
  pairs; the official script does **not** require the repo's own `dataset/demo/...`
  structure for testing, only for *training* — so we can point it straight at your Drive
  folders once filenames are aligned (Section 8).
- **Outputs** → written wherever `--result_dir` points; we redirect this to
  `MyDrive/Results/WaterMamba_Output/`.

This notebook keeps that exact loading logic (YAML → model → checkpoint → per-image
pad/forward/unpad) and only swaps the argparse CLI for direct Colab cells.


## Section 3 — Clone the official repository

In [ ]:
#@title Clone WaterMamba repository
import os

REPO_DIR = "/content/WaterMamba"

if not os.path.exists(REPO_DIR):
    !git clone --quiet https://github.com/Guan-MS/WaterMamba.git "{REPO_DIR}"
else:
    print("Repository already present, skipping clone.")

assert os.path.isdir(REPO_DIR), "Clone failed — repository directory not found."

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)   # ensures the repo's own vendored `basicsr` and `Mamba`
                                    # packages are what get imported, not any PyPI package
os.chdir(REPO_DIR)

print("Repository ready at:", REPO_DIR)
print("\nTop-level structure:")
for entry in sorted(os.listdir(REPO_DIR)):
    print(" -", entry)


## Section 4 — Install dependencies

WaterMamba needs a working `mamba_ssm` + `causal_conv1d` (compiled CUDA selective-scan
kernels). The pinned versions in the repo's README (`causal_conv1d==1.0.0`,
`mamba_ssm==1.1.1`) predate current Colab's CUDA/PyTorch/Python combination and usually have
**no prebuilt wheel** for it, which forces a slow (and often failing) from-source build.

To keep this automatic and robust, the install cell below tries, in order:
1. A plain `pip install` of current `causal-conv1d` / `mamba-ssm` (works if a matching wheel
   exists on PyPI for this exact environment).
2. If that fails: query the GitHub Releases of `Dao-AILab/causal-conv1d` and
   `state-spaces/mamba` directly, find a released wheel whose filename matches this
   Colab's Python / Torch / CUDA build tags, and install that wheel directly.
3. If that *also* fails: fall back to a from-source build with `--no-build-isolation` and
   print a clear diagnostic message rather than silently continuing with a broken import.


In [ ]:
#@title Install PyTorch-adjacent + general dependencies
import sys, subprocess

def pip_install(*args, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd += list(args)
    return subprocess.run(cmd)

# Colab ships a working CUDA-enabled PyTorch already -- we deliberately do NOT reinstall
# torch/torchvision (that risks breaking the preinstalled CUDA toolkit pairing). We only
# add the packages WaterMamba needs on top of it.
pip_install(
    "opencv-python-headless", "Pillow", "numpy<2.0", "scikit-image", "matplotlib",
    "tqdm", "einops", "timm", "natsort", "pytorch_msssim", "PyYAML", "addict",
    "future", "lmdb", "yapf", "pandas", "openpyxl", "packaging", "ninja",
)

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version (torch build):", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
#@title Install causal-conv1d + mamba-ssm (automatic wheel matching, with fallbacks)
import sys, subprocess, re, json, urllib.request

def sh(cmd, check=False):
    print("$", cmd)
    result = subprocess.run(cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result.returncode


def can_import_mamba():
    try:
        import causal_conv1d  # noqa: F401
        import mamba_ssm      # noqa: F401
        return True
    except Exception as e:
        print("mamba_ssm / causal_conv1d not importable yet:", e)
        return False


if can_import_mamba():
    print("causal_conv1d and mamba_ssm already available — skipping install.")
else:
    print("Attempt 1/3: plain pip install of latest causal-conv1d / mamba-ssm ...")
    sh(f"{sys.executable} -m pip install -q causal-conv1d --no-build-isolation")
    sh(f"{sys.executable} -m pip install -q mamba-ssm --no-build-isolation")

if not can_import_mamba():
    print("\nAttempt 2/3: matching a prebuilt wheel from GitHub Releases to this exact "
          "Python / Torch / CUDA build ...")
    import torch
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_ver = torch.__version__.split('+')[0]
    torch_tag = "torch" + ".".join(torch_ver.split(".")[:2])
    cuda_ver = (torch.version.cuda or "").replace(".", "")
    cuda_tag = f"cu{cuda_ver[:3]}" if cuda_ver else None
    print(f"Looking for wheels matching: {py_tag}, {torch_tag}, {cuda_tag}, linux_x86_64, "
          f"cxx11abiFALSE")

    def best_wheel_url(repo):
        try:
            req = urllib.request.Request(
                f"https://api.github.com/repos/{repo}/releases",
                headers={"Accept": "application/vnd.github+json"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                releases = json.loads(resp.read().decode())
        except Exception as e:
            print(f"Could not query releases for {repo}: {e}")
            return None
        candidates = []
        for rel in releases:
            for asset in rel.get("assets", []):
                name = asset["name"]
                if not name.endswith(".whl"):
                    continue
                if py_tag not in name or "linux_x86_64" not in name:
                    continue
                if torch_tag not in name:
                    continue
                if cuda_tag and cuda_tag not in name:
                    continue
                candidates.append((("cxx11abiFALSE" in name), asset["browser_download_url"]))
        if not candidates:
            return None
        # Prefer the cxx11abiFALSE build (matches standard PyPI PyTorch builds).
        candidates.sort(key=lambda t: t[0], reverse=True)
        return candidates[0][1]

    conv1d_url = best_wheel_url("Dao-AILab/causal-conv1d")
    mamba_url = best_wheel_url("state-spaces/mamba")

    if conv1d_url:
        print("Installing causal-conv1d wheel:", conv1d_url)
        sh(f"{sys.executable} -m pip install -q '{conv1d_url}'")
    else:
        print("No matching causal-conv1d wheel found on GitHub Releases.")

    if mamba_url:
        print("Installing mamba-ssm wheel:", mamba_url)
        sh(f"{sys.executable} -m pip install -q '{mamba_url}'")
    else:
        print("No matching mamba-ssm wheel found on GitHub Releases.")

if not can_import_mamba():
    print("\nAttempt 3/3: from-source build (slow, several minutes) ...")
    sh(f"{sys.executable} -m pip install -q causal-conv1d --no-build-isolation --no-cache-dir")
    sh(f"{sys.executable} -m pip install -q mamba-ssm --no-build-isolation --no-cache-dir")

if can_import_mamba():
    import causal_conv1d, mamba_ssm
    print("\ncausal_conv1d and mamba_ssm installed successfully.")
    print("causal_conv1d:", getattr(causal_conv1d, "__version__", "unknown version"))
    print("mamba_ssm:", getattr(mamba_ssm, "__version__", "unknown version"))
else:
    raise RuntimeError(
        "Could not install a working causal_conv1d/mamba_ssm after 3 attempts. "
        "This is almost always a CUDA/PyTorch/Python version mismatch on the current "
        "Colab runtime. Try Runtime > Change runtime type > GPU, then Runtime > "
        "Disconnect and delete runtime to get a fresh environment, and re-run from "
        "Section 4."
    )


In [ ]:
#@title Core imports used throughout the rest of the notebook
import glob
import shutil
import time
import math

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("Core libraries imported successfully.")


## Section 5 — Mount Google Drive

In [ ]:
#@title Mount Google Drive
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive"
assert os.path.isdir(DRIVE_ROOT), "Google Drive did not mount correctly."
print("Google Drive mounted at:", DRIVE_ROOT)


In [ ]:
#@title Configure all project paths (edit ONLY if your folder names differ)
WATERMAMBA_DRIVE_DIR = os.path.join(DRIVE_ROOT, "WaterMamba")
PRETRAINED_PTH        = os.path.join(WATERMAMBA_DRIVE_DIR, "pretrained", "net_g_232000.pth")
CONFIG_YML             = os.path.join(WATERMAMBA_DRIVE_DIR, "configs", "WaterMamba.yml")

DATASET_DIR = os.path.join(DRIVE_ROOT, "Dataset")
RAW_DIR     = os.path.join(DATASET_DIR, "gc_real")   # YOUR raw underwater images
GT_DIR      = os.path.join(DATASET_DIR, "ce_real")   # YOUR ground-truth images

RESULTS_DIR = os.path.join(DRIVE_ROOT, "Results", "WaterMamba_Output")

# Fallback Google Drive shared-folder links, used automatically only if the paths above
# are missing, so nothing needs to be done manually.
PRETRAINED_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1FDrSsU9Sn5lg1gKQ1buxfgbpaYmWfH68"
DATASET_DRIVE_FOLDER_URL    = "https://drive.google.com/drive/folders/1-6XfkD1hXcZdT3agKS979lUwwTuJmMnp"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Pretrained weights :", PRETRAINED_PTH)
print("Config YAML         :", CONFIG_YML)
print("Raw images dir       :", RAW_DIR)
print("GT images dir         :", GT_DIR)
print("Results dir            :", RESULTS_DIR)


## Section 6 — Verify environment

In [ ]:
#@title Environment diagnostics
import platform, torch

print("=" * 55)
print(" ENVIRONMENT")
print("=" * 55)
print("Python version   :", platform.python_version())
print("PyTorch version  :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version     :", torch.version.cuda)
    print("GPU name         :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU. "
          "WaterMamba inference on CPU will be extremely slow.")

try:
    import google.colab
    print("Running in       : Google Colab")
except ImportError:
    print("Running in       : NOT Google Colab (some cells may need adjustment)")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device     :", DEVICE)


## Section 7 — Load WaterMamba (official architecture + your checkpoint)

This reuses the exact loading sequence from the official `Mamba/test.py`:
`yaml.load(...)` → `WaterMamba(**network_g)` → `torch.load(weights)["params"]` →
`load_state_dict(...)`.

In [ ]:
#@title Verify pretrained weights + YAML exist (auto-download from Drive link as fallback)
import gdown, glob, shutil

def file_ok(p):
    return os.path.isfile(p) and os.path.getsize(p) > 0

if not (file_ok(PRETRAINED_PTH) and file_ok(CONFIG_YML)):
    print("Pretrained model / config not found at the expected path — attempting "
          "automatic download from the shared Google Drive folder as a fallback...")
    os.makedirs(WATERMAMBA_DRIVE_DIR, exist_ok=True)
    tmp_dl = "/content/_watermamba_pretrained_dl"
    try:
        gdown.download_folder(url=PRETRAINED_DRIVE_FOLDER_URL, output=tmp_dl,
                               quiet=False, use_cookies=False)
        # Flatten: find the .pth and .yml wherever gdown put them and copy to expected paths.
        found_pth = glob.glob(os.path.join(tmp_dl, "**", "*.pth"), recursive=True)
        found_yml = glob.glob(os.path.join(tmp_dl, "**", "*.yml"), recursive=True) + \
                    glob.glob(os.path.join(tmp_dl, "**", "*.yaml"), recursive=True)
        if found_pth:
            os.makedirs(os.path.dirname(PRETRAINED_PTH), exist_ok=True)
            shutil.copy(found_pth[0], PRETRAINED_PTH)
        if found_yml:
            os.makedirs(os.path.dirname(CONFIG_YML), exist_ok=True)
            shutil.copy(found_yml[0], CONFIG_YML)
    except Exception as e:
        print("Automatic download failed:", e)

# If the YAML still isn't present, fall back to the copy already sitting inside the
# cloned repository itself (Mamba/Options/WaterMamba.yml) rather than failing outright.
if not file_ok(CONFIG_YML):
    repo_yaml = os.path.join(REPO_DIR, "Mamba", "Options", "WaterMamba.yml")
    if file_ok(repo_yaml):
        print(f"Using the repository's own config file as a fallback: {repo_yaml}")
        CONFIG_YML = repo_yaml

if not file_ok(PRETRAINED_PTH):
    raise FileNotFoundError(
        f"Could not find pretrained weights at '{PRETRAINED_PTH}'.\n"
        f"Please make sure your checkpoint is placed at:\n"
        f"  MyDrive/WaterMamba/pretrained/net_g_232000.pth\n"
        f"or that the shared Drive folder link is accessible to your account."
    )
if not file_ok(CONFIG_YML):
    raise FileNotFoundError(
        f"Could not find a WaterMamba.yml config at '{CONFIG_YML}' or inside the "
        f"cloned repository. Please place your config at:\n"
        f"  MyDrive/WaterMamba/configs/WaterMamba.yml"
    )

print("Pretrained weights verified:", PRETRAINED_PTH,
      f"({os.path.getsize(PRETRAINED_PTH) / 1e6:.1f} MB)")
print("Config YAML verified       :", CONFIG_YML)


In [ ]:
#@title Build the official WaterMamba model and load your checkpoint
import yaml
try:
    from yaml import CLoader as Loader
except ImportError:
    from yaml import Loader

# Reuses the repo's own architecture class -- not reimplemented.
from basicsr.models.archs.WaterMamba_arch import WaterMamba

with open(CONFIG_YML, mode='r') as f:
    cfg = yaml.load(f, Loader=Loader)

network_cfg = dict(cfg['network_g'])
arch_type = network_cfg.pop('type')   # matches test.py: x['network_g'].pop('type')
print(f"Config network type: {arch_type}")
print("Network hyperparameters:", network_cfg)

model = WaterMamba(**network_cfg)

checkpoint = torch.load(PRETRAINED_PTH, map_location=DEVICE)
state_dict = checkpoint["params"] if "params" in checkpoint else checkpoint
missing, unexpected = model.load_state_dict(state_dict, strict=True)

model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"\n [*] WaterMamba loaded successfully.")
print(f" [*] Total parameters: {n_params / 1e6:.2f} M")
print(f" [*] Weights file    : {os.path.basename(PRETRAINED_PTH)}")
print(f" [*] Device          : {DEVICE}")


## Section 8 — Dataset preparation

WaterMamba's own `test.py` expects the ground-truth filename for each raw image to be
**exactly** the same string (`target_dir/<same filename as raw>`). Your raw/GT files may
not share identical filenames or extensions, so this step matches them by base filename
and copies them into two temp folders with unified names — automatically, no manual
renaming.

In [ ]:
#@title Verify dataset folders and match raw <-> GT pairs by filename
def list_images(folder):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.PNG", "*.JPG", "*.JPEG", "*.BMP")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(folder, e)))
    return sorted(files)

if not os.path.isdir(RAW_DIR):
    raise FileNotFoundError(f"Raw image folder not found: {RAW_DIR}")
if not os.path.isdir(GT_DIR):
    raise FileNotFoundError(f"Ground-truth folder not found: {GT_DIR}")

raw_files = list_images(RAW_DIR)
gt_files  = list_images(GT_DIR)

if len(raw_files) == 0:
    raise FileNotFoundError(f"No images found in raw folder: {RAW_DIR}")
if len(gt_files) == 0:
    raise FileNotFoundError(f"No images found in ground-truth folder: {GT_DIR}")

raw_index = {os.path.splitext(os.path.basename(f))[0]: f for f in raw_files}
gt_index  = {os.path.splitext(os.path.basename(f))[0]: f for f in gt_files}

common_keys = sorted(set(raw_index.keys()) & set(gt_index.keys()))
missing_gt  = sorted(set(raw_index.keys()) - set(gt_index.keys()))
missing_raw = sorted(set(gt_index.keys()) - set(raw_index.keys()))

if missing_gt:
    print(f"WARNING: {len(missing_gt)} raw image(s) have no matching ground truth and "
          f"will be skipped for evaluation, e.g.: {missing_gt[:5]}")
if missing_raw:
    print(f"WARNING: {len(missing_raw)} ground-truth image(s) have no matching raw "
          f"image and will be ignored, e.g.: {missing_raw[:5]}")

if len(common_keys) == 0:
    raise FileNotFoundError(
        "No filename matches at all between raw and ground-truth folders — please "
        "check that corresponding files share the same base filename."
    )

paired_files = [(raw_index[k], gt_index[k]) for k in common_keys]

print(f"Total raw images found        : {len(raw_files)}")
print(f"Total ground-truth images found: {len(gt_files)}")
print(f"Matched raw/GT pairs           : {len(paired_files)}")
print("\nFirst few matched image names:")
for k in common_keys[:10]:
    print(" -", k)

sizes = [os.path.getsize(f) for f, _ in paired_files]
print(f"\nDataset statistics: avg raw file size = {sum(sizes)/len(sizes)/1024:.1f} KB, "
      f"min = {min(sizes)/1024:.1f} KB, max = {max(sizes)/1024:.1f} KB")


In [ ]:
#@title Build unified-filename input/target folders expected by test.py's logic
WORK_DIR = "/content/_wm_work"
INPUT_DIR = os.path.join(WORK_DIR, "input")
TARGET_DIR = os.path.join(WORK_DIR, "target")

for d in (INPUT_DIR, TARGET_DIR):
    os.makedirs(d, exist_ok=True)

import cv2

skipped = []
final_pairs = []
for raw_path, gt_path in tqdm(paired_files, desc="Preparing matched input/target pairs"):
    stem = os.path.splitext(os.path.basename(raw_path))[0]
    out_name = stem + ".png"
    try:
        raw_img = cv2.imread(raw_path, cv2.IMREAD_COLOR)
        gt_img = cv2.imread(gt_path, cv2.IMREAD_COLOR)
        if raw_img is None or gt_img is None:
            raise ValueError("Unreadable / corrupt image file")
        cv2.imwrite(os.path.join(INPUT_DIR, out_name), raw_img)
        cv2.imwrite(os.path.join(TARGET_DIR, out_name), gt_img)
        final_pairs.append((os.path.join(INPUT_DIR, out_name),
                             os.path.join(TARGET_DIR, out_name), stem))
    except Exception as e:
        print(f"Skipping '{stem}' due to error: {e}")
        skipped.append(stem)

print(f"\nPrepared {len(final_pairs)} matched pairs in {WORK_DIR} "
      f"({len(skipped)} skipped due to errors).")


## Section 9 — Run inference

Padding, forward pass, and unpadding below follow the official `test.py` exactly
(`factor = 8`, reflect-padding to a multiple of 8, crop back to original size, clamp to
[0, 1]). A tiled fallback is added only as an automatic recovery path if a single very
large image runs out of GPU memory — normal-sized images never touch it.

In [ ]:
#@title Load-image helper (reuses repo's Mamba.utils if available, else an equivalent)
try:
    import Mamba.utils as repo_utils
    def load_img(path):
        return repo_utils.load_img(path)
    def save_img(path, img):
        repo_utils.save_img(path, img)
    print("Using the repository's own Mamba.utils.load_img / save_img.")
except Exception as e:
    print(f"Could not import Mamba.utils ({e}); using an equivalent RGB load/save helper.")
    def load_img(path):
        return cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    def save_img(path, img):
        cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


In [ ]:
#@title Run WaterMamba inference over every matched image
import torch.nn.functional as F
from skimage import img_as_ubyte
import time

FACTOR = 8          # matches test.py: encoder downsamples 3x, so H/W must be multiples of 8
TILE_SIZE = 512      # only used as an OOM fallback for very large images
TILE_OVERLAP = 32

def run_single(input_tensor):
    """Exact pad/forward/unpad logic from the official test.py."""
    h, w = input_tensor.shape[2], input_tensor.shape[3]
    H = ((h + FACTOR) // FACTOR) * FACTOR
    W = ((w + FACTOR) // FACTOR) * FACTOR
    padh = H - h if h % FACTOR != 0 else 0
    padw = W - w if w % FACTOR != 0 else 0
    padded = F.pad(input_tensor, (0, padw, 0, padh), 'reflect')
    with torch.no_grad():
        restored = model(padded)
    restored = restored[:, :, :h, :w]
    return torch.clamp(restored, 0, 1)


def run_tiled(input_tensor, tile=TILE_SIZE, overlap=TILE_OVERLAP):
    """Fallback for images too large to fit in GPU memory in one pass."""
    b, c, h, w = input_tensor.shape
    output = torch.zeros_like(input_tensor)
    weight = torch.zeros((1, 1, h, w), device=input_tensor.device)
    stride = tile - overlap
    for y in range(0, h, stride):
        for x in range(0, w, stride):
            y2, x2 = min(y + tile, h), min(x + tile, w)
            y1, x1 = max(y2 - tile, 0), max(x2 - tile, 0)
            patch = input_tensor[:, :, y1:y2, x1:x2]
            out_patch = run_single(patch)
            output[:, :, y1:y2, x1:x2] += out_patch
            weight[:, :, y1:y2, x1:x2] += 1.0
    return torch.clamp(output / weight.clamp(min=1.0), 0, 1)


enhanced_records = []
failed_records = []
start_time = time.time()

for raw_path, gt_path, stem in tqdm(final_pairs, desc="Running WaterMamba inference"):
    out_name = stem + ".png"
    try:
        img = np.float32(load_img(raw_path)) / 255.0
        img_t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        try:
            restored = run_single(img_t)
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on '{stem}' at full resolution — retrying with tiled inference.")
            torch.cuda.empty_cache()
            restored = run_tiled(img_t)

        restored_np = restored.detach().cpu().permute(0, 2, 3, 1).squeeze(0).numpy()
        out_path = os.path.join(RESULTS_DIR, out_name)
        save_img(out_path, img_as_ubyte(restored_np))

        enhanced_records.append({
            "name": stem, "raw_path": raw_path, "gt_path": gt_path, "output_path": out_path,
        })
    except Exception as e:
        print(f"Inference failed on '{stem}': {e}")
        failed_records.append(stem)

elapsed = time.time() - start_time
print(f"\nInference complete: {len(enhanced_records)} images enhanced and saved to "
      f"{RESULTS_DIR} in {elapsed:.1f}s")
if failed_records:
    print(f"{len(failed_records)} image(s) failed during inference: {failed_records}")


## Section 10 — Visualize Raw → Enhanced → Ground-Truth

In [ ]:
#@title Show side-by-side comparisons for a handful of sample images
N_SAMPLES = min(5, len(enhanced_records))
sample_records = enhanced_records[:N_SAMPLES]

if N_SAMPLES == 0:
    print("No successfully enhanced images available to visualize.")
else:
    fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(12, 4 * N_SAMPLES))
    if N_SAMPLES == 1:
        axes = axes[np.newaxis, :]

    for row, rec in enumerate(sample_records):
        raw_img = load_img(rec["raw_path"])
        out_img = load_img(rec["output_path"])
        gt_img  = load_img(rec["gt_path"])

        for col, (img, title) in enumerate(zip(
                [raw_img, out_img, gt_img],
                ["Raw", "WaterMamba Enhanced", "Ground Truth"])):
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{rec['name']} — {title}", fontsize=10)
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()


## Section 11 — Evaluate against ground truth (PSNR, SSIM, UIQM, UCIQE)

PSNR/SSIM are full-reference (compared against `Dataset/ce_real`). UIQM (Panetta et al.,
2016) and UCIQE (Yang & Sowmya, 2015) are no-reference metrics computed on the enhanced
output only, standard in underwater image enhancement literature.

In [ ]:
#@title No-reference UIQM / UCIQE metric implementations
import math

def _uicm(img_rgb):
    R = img_rgb[..., 0].astype(np.float64)
    G = img_rgb[..., 1].astype(np.float64)
    B = img_rgb[..., 2].astype(np.float64)
    RG = R - G
    YB = 0.5 * (R + G) - B

    def _mu_alpha(x, alpha=0.1):
        x = np.sort(x.flatten())
        n = len(x)
        low = int(np.floor(alpha * n))
        high = int(np.ceil((1 - alpha) * n))
        trimmed = x[low:high] if high > low else x
        return trimmed.mean() if trimmed.size else x.mean()

    rg_mu, rg_var = _mu_alpha(RG), np.var(RG)
    yb_mu, yb_var = _mu_alpha(YB), np.var(YB)
    return -0.0268 * math.sqrt(rg_mu ** 2 + yb_mu ** 2) + 0.1586 * math.sqrt(rg_var + yb_var)


def _uism(img_rgb):
    weights = (0.299, 0.587, 0.114)
    sm_total = 0.0
    for c in range(3):
        channel = img_rgb[..., c].astype(np.float64)
        sobelx = cv2.Sobel(channel, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(channel, cv2.CV_64F, 0, 1, ksize=3)
        edge_map = np.sqrt(sobelx ** 2 + sobely ** 2)
        sm_total += weights[c] * edge_map.mean()
    return sm_total


def _uiconm(img_rgb, block_size=8, alpha=1.0):
    # Bounded log-AMEE style contrast measure (weighting by contrast**alpha keeps every
    # block's contribution bounded, instead of blowing up toward -inf on flat blocks).
    gray = cv2.cvtColor(img_rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
    h, w = gray.shape
    vals = []
    for y in range(0, h - h % block_size, block_size):
        for x in range(0, w - w % block_size, block_size):
            block = gray[y:y + block_size, x:x + block_size]
            mx, mn = block.max(), block.min()
            denom = mx + mn
            if denom <= 1e-8:
                continue
            ratio = (mx - mn) / denom
            if ratio <= 1e-8:
                vals.append(0.0)
            else:
                vals.append(alpha * (ratio ** alpha) * math.log(ratio))
    return float(np.mean(vals)) if vals else 0.0


def compute_uiqm(img_rgb_uint8):
    c1, c2, c3 = 0.0282, 0.2953, 3.5753
    return float(c1 * _uicm(img_rgb_uint8) + c2 * _uism(img_rgb_uint8) + c3 * _uiconm(img_rgb_uint8))


def compute_uciqe(img_rgb_uint8):
    # OpenCV's 8-bit Lab output encodes L in [0,255] (true range [0,100]) and a/b shifted
    # by +128 (true range roughly [-127,127], centered on 0 for a neutral pixel); both
    # must be corrected before computing chroma/contrast.
    lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2LAB).astype(np.float64)
    L = lab[..., 0] * (100.0 / 255.0)
    a = lab[..., 1] - 128.0
    b = lab[..., 2] - 128.0

    chroma = np.sqrt(a ** 2 + b ** 2)
    sigma_c = chroma.std()

    l_flat = np.sort(L.flatten())
    n = len(l_flat)
    top = l_flat[int(0.99 * n):] if int(0.99 * n) < n else l_flat[-1:]
    bottom = l_flat[:max(int(0.01 * n), 1)]
    con_l = (top.mean() if top.size else L.max()) - (bottom.mean() if bottom.size else L.min())

    hsv = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2HSV).astype(np.float64)
    mu_s = (hsv[..., 1] / 255.0).mean()   # OpenCV's HSV saturation is [0,255], UCIQE expects [0,1]

    c1, c2, c3 = 0.4680, 0.2745, 0.2576
    return float(c1 * sigma_c + c2 * con_l + c3 * mu_s)

print("UIQM / UCIQE implementations ready.")


In [ ]:
#@title Compute PSNR / SSIM / UIQM / UCIQE for every image and build the results table
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim

rows = []
for rec in tqdm(enhanced_records, desc="Evaluating metrics"):
    try:
        enh_rgb = load_img(rec["output_path"])
        gt_rgb  = load_img(rec["gt_path"])

        if enh_rgb.shape != gt_rgb.shape:
            gt_rgb = cv2.resize(gt_rgb, (enh_rgb.shape[1], enh_rgb.shape[0]),
                                 interpolation=cv2.INTER_AREA)

        psnr_val = sk_psnr(gt_rgb, enh_rgb, data_range=255)
        ssim_val = sk_ssim(gt_rgb, enh_rgb, channel_axis=2, data_range=255)
        uiqm_val = compute_uiqm(enh_rgb)
        uciqe_val = compute_uciqe(enh_rgb)

        rows.append({
            "image": rec["name"], "PSNR": psnr_val, "SSIM": ssim_val,
            "UIQM": uiqm_val, "UCIQE": uciqe_val,
        })
    except Exception as e:
        print(f"Metric computation failed for '{rec['name']}': {e}")

metrics_df = pd.DataFrame(rows)
if metrics_df.empty:
    raise RuntimeError("No metrics could be computed — check that enhanced/GT images "
                        "loaded correctly in the previous sections.")

display_df = metrics_df.copy()
for col in ["PSNR", "SSIM", "UIQM", "UCIQE"]:
    display_df[col] = display_df[col].round(4)
print(display_df.to_string(index=False))


In [ ]:
#@title Export metrics.csv and metrics.xlsx to Google Drive
csv_path = os.path.join(RESULTS_DIR, "metrics.csv")
xlsx_path = os.path.join(RESULTS_DIR, "metrics.xlsx")

metrics_df.to_csv(csv_path, index=False)

summary_row = {
    "image": "AVERAGE",
    "PSNR": metrics_df["PSNR"].mean(),
    "SSIM": metrics_df["SSIM"].mean(),
    "UIQM": metrics_df["UIQM"].mean(),
    "UCIQE": metrics_df["UCIQE"].mean(),
}
export_df = pd.concat([metrics_df, pd.DataFrame([summary_row])], ignore_index=True)
export_df.to_excel(xlsx_path, index=False)

print("Exported:")
print(" -", csv_path)
print(" -", xlsx_path)


## Section 12 — Summary

In [ ]:
#@title Final run summary
avg_psnr = metrics_df["PSNR"].mean()
avg_ssim = metrics_df["SSIM"].mean()
avg_uiqm = metrics_df["UIQM"].mean()
avg_uciqe = metrics_df["UCIQE"].mean()

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"

print("=" * 55)
print(" WATERMAMBA INFERENCE & BENCHMARKING — SUMMARY")
print("=" * 55)
print(f" Images processed successfully : {len(enhanced_records)}")
print(f" Images failed                 : {len(failed_records)}")
print(f" Average PSNR                  : {avg_psnr:.4f} dB")
print(f" Average SSIM                  : {avg_ssim:.4f}")
print(f" Average UIQM                  : {avg_uiqm:.4f}")
print(f" Average UCIQE                 : {avg_uciqe:.4f}")
print(f" Total inference time          : {elapsed:.1f} s")
print(f" GPU used                      : {gpu_name}")
print("-" * 55)
print(f" Enhanced images  : {RESULTS_DIR}")
print(f" metrics.csv      : {csv_path}")
print(f" metrics.xlsx     : {xlsx_path}")
print("=" * 55)
